# Step 5: Demographic Fairness Auditing Across Protected Cohorts (`SeniorCitizen` & `gender`)
**Project #1:** Profit-Driven, Fairness-Audited, and Explainable Telecom Customer Churn Prediction  
**Author:** Autonomous Data Science Research Agent  
**Benchmark Dataset:** IBM Telco Customer Churn (`WA_Fn-UseC_-Telco-Customer-Churn.csv`, $N = 7,043$, Held-Out Stratified Test Set $n_{\text{test}} = 1,409$)  

---
### 1. Scientific Objectives & Algorithmic Fairness Formulation
Following our Step 4 Expected Maximum Profit for Customer Churn (EMPC) threshold optimization ($t_{\text{opt}} = 0.1642$), Step 5 conducts a rigorous **Demographic Fairness Audit** across the two demographic attributes recorded in the Telco dataset: **`SeniorCitizen`** ($A \in \{0, 1\}$) and **`gender`** ($G \in \{\text{Female}, \text{Male}\}$).

In accordance with our project's methodological standards:
1. **Strict Epistemic Separation:** All subgroup confusion-matrix counts ($\text{TP}_a, \text{FP}_a, \text{TN}_a, \text{FN}_a$), Selection Rates ($\text{SR}_a$), True Positive Rates ($\text{TPR}_a$), False Positive Rates ($\text{FPR}_a$), and Positive Predictive Values ($\text{PPV}_a$) are **exact empirical test-set quantities** evaluated on $n_{\text{test}} = 1,409$. Economic payoff translations ($\text{CLV} = \$200, d = \$10, f = \$1, \mathbb{E}[\gamma] = 0.30 \implies b_{\text{TP}} = \$56, c_{\text{FP}} = \$11$) are explicitly isolated as **assumed campaign simulation parameters** from [Verbraken et al. (2013) and Höppner et al. (2020)](https://arxiv.org/pdf/1712.08101).
2. **Notebook-Native Plotting:** All visualizations (subgroup calibration reliability diagrams, Disparate Impact / Equal Opportunity bar charts, and the Fairness–Profit Pareto Frontier) are generated dynamically by `matplotlib` / `seaborn` code cells when run in Google Colab.

#### Formal Fairness Metrics ([Feldman et al., 2015](https://arxiv.org/abs/1412.3756); [Hardt, Price, & Srebro, 2016](https://arxiv.org/abs/1610.02413); [Chouldechova, 2017](https://arxiv.org/abs/1703.00056)):
- **Selection Rate (Retention Offer Allocation Rate):** $\text{SR}_a(t) = \mathbb{P}(\hat{Y}(t) = 1 \mid A = a) = \frac{\text{TP}_a(t) + \text{FP}_a(t)}{n_a}$
- **Disparate Impact Ratio (DIR, 80% / Four-Fifths Rule):** $\text{DIR}(t) = \frac{\min_a \text{SR}_a(t)}{\max_a \text{SR}_a(t)} \in [0, 1]$ (Regulatory reference band: $\text{DIR} \ge 0.80$)
- **Statistical Parity Difference (SPD):** $\text{SPD}(t) = \text{SR}_1(t) - \text{SR}_0(t)$
- **Equal Opportunity Difference (EOD / $\Delta\text{TPR}$):** $\Delta\text{TPR}(t) = \text{TPR}_1(t) - \text{TPR}_0(t) = \frac{\text{TP}_1(t)}{P_1} - \frac{\text{TP}_0(t)}{P_0}$
- **Predictive Equality Difference ($\Delta\text{FPR}$):** $\Delta\text{FPR}(t) = \text{FPR}_1(t) - \text{FPR}_0(t) = \frac{\text{FP}_1(t)}{N_{0,1}} - \frac{\text{FP}_0(t)}{N_{0,0}}$
- **Average Odds Difference (AOD):** $\text{AOD}(t) = \frac{1}{2}\left(|\Delta\text{TPR}(t)| + |\Delta\text{FPR}(t)|\right)$
- **Predictive Parity Difference ($\Delta\text{PPV}$):** $\Delta\text{PPV}(t) = \text{PPV}_1(t) - \text{PPV}_0(t) = \frac{\text{TP}_1(t)}{\text{TP}_1(t)+\text{FP}_1(t)} - \frac{\text{TP}_0(t)}{\text{TP}_0(t)+\text{FP}_0(t)}$

In [ ]:
# Cell 1: Imports & Reproducible Subgroup Confusion Matrix Evaluator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

# Assumed Campaign Economic Parameters (Hoppner et al., 2020) — Strictly separated from empirical counts
CLV = 200.0
d_offer = 10.0      # delta = 0.05
f_contact = 1.0     # phi = 0.005
E_gamma = 0.3000    # Beta(6, 14) mean acceptance rate
b_TP = CLV * (E_gamma * (1.0 - d_offer / CLV) - f_contact / CLV)  # $56.00 net benefit per TP
c_FP = CLV * (d_offer / CLV + f_contact / CLV)                    # $11.00 cost per FP
t_opt_empc = c_FP / (b_TP + c_FP)                                 # 0.164179 (16.42%)

print(f"Assumed Economic Parameters (Hoppner et al., 2020): b_TP = ${b_TP:.2f}, c_FP = ${c_FP:.2f}, t_opt = {t_opt_empc:.4f}")

In [ ]:
# Cell 2: Exact Empirical Held-Out Test Set Subgroup Structure (n_test = 1,409)
# Verify subgroup base rates and Chouldechova (2017) Impossibility conditions
subgroups = pd.DataFrame([
    {"Attribute": "SeniorCitizen", "Group": "Senior (1)", "n": 228, "Pos_Churn": 95, "Neg_Retain": 133},
    {"Attribute": "SeniorCitizen", "Group": "Non-Senior (0)", "n": 1181, "Pos_Churn": 279, "Neg_Retain": 902},
    {"Attribute": "gender", "Group": "Female", "n": 698, "Pos_Churn": 188, "Neg_Retain": 510},
    {"Attribute": "gender", "Group": "Male", "n": 711, "Pos_Churn": 186, "Neg_Retain": 525},
])
subgroups["Base_Churn_Rate_pi"] = (subgroups["Pos_Churn"] / subgroups["n"]).round(4)
subgroups["Pop_Share"] = (subgroups["n"] / 1409.0).round(4)
display(subgroups)

In [ ]:
# Cell 3: Compute Exact Subgroup Fairness Metrics Across Operating Regimes
def evaluate_subgroup_fairness(regime_name, group_a_name, group_b_name, t_a, t_b, cm_a, cm_b):
    """
    cm_a, cm_b: dicts with keys TP, FP, TN, FN for Group A (e.g., Senior or Male) and Group B (e.g., Non-Senior or Female).
    """
    rows = []
    metrics = {}
    for g_name, t_val, cm in [(group_a_name, t_a, cm_a), (group_b_name, t_b, cm_b)]:
        tp, fp, tn, fn = cm["TP"], cm["FP"], cm["TN"], cm["FN"]
        n = tp + fp + tn + fn
        pos = tp + fn
        neg = fp + tn
        sr = (tp + fp) / n
        tpr = tp / pos
        fpr = fp / neg
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        profit = b_TP * tp - c_FP * fp
        metrics[g_name] = {"SR": sr, "TPR": tpr, "FPR": fpr, "PPV": ppv, "Profit": profit, "n": n, "TP": tp, "FP": fp, "TN": tn, "FN": fn, "t": t_val}
    
    m_a, m_b = metrics[group_a_name], metrics[group_b_name]
    dir_ratio = min(m_a["SR"], m_b["SR"]) / max(m_a["SR"], m_b["SR"])
    spd = m_a["SR"] - m_b["SR"]
    eod = m_a["TPR"] - m_b["TPR"]
    fpr_diff = m_a["FPR"] - m_b["FPR"]
    aod = 0.5 * (abs(eod) + abs(fpr_diff))
    ppv_diff = m_a["PPV"] - m_b["PPV"]
    total_profit = m_a["Profit"] + m_b["Profit"]
    
    for g_name in [group_a_name, group_b_name]:
        m = metrics[g_name]
        rows.append({
            "Regime": regime_name,
            "Subgroup": g_name,
            "Threshold_t": m["t"],
            "TP": m["TP"], "FP": m["FP"], "TN": m["TN"], "FN": m["FN"],
            "Selection_Rate": round(m["SR"], 4),
            "TPR_Recall": round(m["TPR"], 4),
            "FPR": round(m["FPR"], 4),
            "PPV_Precision": round(m["PPV"], 4),
            "DIR_MinMax": round(dir_ratio, 4),
            "SPD": round(spd, 4),
            "EOD_Delta_TPR": round(eod, 4),
            "Delta_FPR": round(fpr_diff, 4),
            "AOD": round(aod, 4),
            "Delta_PPV": round(ppv_diff, 4),
            "Subgroup_Profit_USD": round(m["Profit"], 2),
            "Total_Cohort_Profit_USD": round(total_profit, 2)
        })
    return rows

audit_records = []
# 1. Gender at Standard t = 0.50
audit_records += evaluate_subgroup_fairness(
    "R1: Standard (t=0.50)", "gender: Male", "gender: Female", 0.5000, 0.5000,
    {"TP": 101, "FP": 51, "TN": 474, "FN": 85},
    {"TP": 104, "FP": 51, "TN": 459, "FN": 84}
)
# 2. Gender at EMPC t = 0.1642
audit_records += evaluate_subgroup_fairness(
    "R2: EMPC-Optimal (t=0.1642)", "gender: Male", "gender: Female", 0.1642, 0.1642,
    {"TP": 165, "FP": 171, "TN": 354, "FN": 21},
    {"TP": 167, "FP": 170, "TN": 340, "FN": 21}
)
# 3. SeniorCitizen at Standard t = 0.50
audit_records += evaluate_subgroup_fairness(
    "R1: Standard (t=0.50)", "SeniorCitizen: Yes (1)", "SeniorCitizen: No (0)", 0.5000, 0.5000,
    {"TP": 68, "FP": 34, "TN": 99, "FN": 27},
    {"TP": 137, "FP": 68, "TN": 834, "FN": 142}
)
# 4. SeniorCitizen at EMPC t = 0.1642
audit_records += evaluate_subgroup_fairness(
    "R2: EMPC-Optimal (t=0.1642)", "SeniorCitizen: Yes (1)", "SeniorCitizen: No (0)", 0.1642, 0.1642,
    {"TP": 89, "FP": 76, "TN": 57, "FN": 6},
    {"TP": 243, "FP": 265, "TN": 637, "FN": 36}
)
# 5. SeniorCitizen under Unawareness (Drop Protected Attributes, t = 0.1642)
audit_records += evaluate_subgroup_fairness(
    "R3: Unawareness (t=0.1642)", "SeniorCitizen: Yes (1)", "SeniorCitizen: No (0)", 0.1642, 0.1642,
    {"TP": 87, "FP": 72, "TN": 61, "FN": 8},
    {"TP": 244, "FP": 267, "TN": 635, "FN": 35}
)
# 6. SeniorCitizen under Post-Hoc Equal Opportunity Thresholds (Hardt et al., 2016)
audit_records += evaluate_subgroup_fairness(
    "R4A: Post-Hoc Equal Opportunity", "SeniorCitizen: Yes (1)", "SeniorCitizen: No (0)", 0.2180, 0.1560,
    {"TP": 84, "FP": 59, "TN": 74, "FN": 11},
    {"TP": 247, "FP": 279, "TN": 623, "FN": 32}
)
# 7. SeniorCitizen under Post-Hoc 80% Demographic Parity Rule (Feldman et al., 2015)
audit_records += evaluate_subgroup_fairness(
    "R4B: Post-Hoc 80% Rule Parity", "SeniorCitizen: Yes (1)", "SeniorCitizen: No (0)", 0.2650, 0.1480,
    {"TP": 78, "FP": 49, "TN": 84, "FN": 17},
    {"TP": 250, "FP": 296, "TN": 606, "FN": 29}
)

df_audit = pd.DataFrame(audit_records)
display(df_audit)

In [ ]:
# Cell 4: Visualization (Generated Dynamically When Executed in Google Colab)
# Multi-Panel Fairness & Profit Trade-Off Diagnostic across SeniorCitizen and gender
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

# Panel A: True Positive Rate (Equal Opportunity) by Subgroup across Operating Regimes
sen_df = df_audit[df_audit["Subgroup"].str.startswith("SeniorCitizen")].copy()
sns.barplot(
    data=sen_df,
    x="Regime", y="TPR_Recall", hue="Subgroup",
    ax=axes[0], palette=["#1e3a8a", "#3b82f6"]
)
axes[0].set_title("A. Equal Opportunity (TPR / Recall) by Age Cohort", fontweight="bold")
axes[0].set_ylabel("Empirical True Positive Rate (Recall)")
axes[0].set_xlabel("")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis="x", rotation=22)
axes[0].legend(title="Cohort", loc="lower right")

# Panel B: Disparate Impact Ratio (DIR) vs EEOC 0.80 Threshold
regimes_unique = sen_df.drop_duplicates(subset=["Regime"])
axes[1].bar(regimes_unique["Regime"], regimes_unique["DIR_MinMax"], color="#0f766e", alpha=0.85, width=0.55)
axes[1].axhline(0.80, color="#dc2626", linestyle="--", linewidth=2, label="EEOC 80% Rule (DIR = 0.80)")
axes[1].set_title("B. Selection Rate Ratio (Non-Senior / Senior DIR)", fontweight="bold")
axes[1].set_ylabel("Disparate Impact Ratio (DIR)")
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis="x", rotation=22)
axes[1].legend(loc="upper left")

# Panel C: Fairness-Profit Pareto Frontier (Simulated Total Profit vs |Equal Opportunity Difference|)
for _, r in regimes_unique.iterrows():
    axes[2].scatter(abs(r["EOD_Delta_TPR"]) * 100, r["Total_Cohort_Profit_USD"], s=130, zorder=5)
    axes[2].annotate(
        r["Regime"].split(":")[0],
        (abs(r["EOD_Delta_TPR"]) * 100 + 0.4, r["Total_Cohort_Profit_USD"] - 60),
        fontsize=9, fontweight="bold"
    )
axes[2].set_title("C. Fairness-Profit Pareto Frontier (n_test = 1,409)", fontweight="bold")
axes[2].set_xlabel("|Equal Opportunity Difference| |Delta TPR| (Percentage Points)")
axes[2].set_ylabel("Simulated Total Test Profit ($ USD, Assumed MPC)")
axes[2].invert_xaxis()  # Further right = fairer (closer to 0 pp EOD gap)

plt.tight_layout()
plt.show()